# Using PyTorch with TensorRT through ONNX:

TensorRT is a great way to take a trained PyTorch model and optimize it to run more efficiently during inference on an NVIDIA GPU.

One approach to convert a PyTorch model to TensorRT is to export a PyTorch model to ONNX (an open format exchange for deep learning models) and then convert into a TensorRT engine. Essentially, we will follow this path to convert and deploy our model:

![PyTorch+ONNX](./images/pytorch_onnx.png)

Both PyTorch and TensorFlow models can be exported to ONNX, as well as many other frameworks. This allows models created using either framework to flow into common downstream pipelines.

To get started, let's take a well-known computer vision model and follow five key steps to deploy it to the TensorRT Python runtime:

1. __What format should I save my model in?__
2. __What batch size(s) am I running inference at?__
3. __What precision am I running inference at?__
4. __What TensorRT path am I using to convert my model?__
5. __What runtime am I targeting?__

## 1. What format should I save my model in?

We will use pretrained ResNet50. The exported ONNX graph will specify its precision explicitly before TensorRT builds the engine.

In [ ]:
import copy
import torchvision.models as models
import torch
import torch.onnx

resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1, progress=False).eval()

When saving a model to ONNX, PyTorch requires a test batch in proper shape and format. We pick a batch size:

In [ ]:
BATCH_SIZE = 32
USE_FP16 = True
torch_dtype = torch.float16 if USE_FP16 else torch.float32

dummy_input = torch.randn(BATCH_SIZE, 3, 224, 224, dtype=torch_dtype)

Next, we will export the model using the dummy input batch:

In [ ]:
# Set precision in the model, not in a TensorRT builder flag.
export_model = copy.deepcopy(resnet50).to(dtype=torch_dtype)
torch.onnx.export(
    export_model, dummy_input, "resnet50_pytorch.onnx",
    input_names=["images"], output_names=["logits"],
    opset_version=17, dynamo=False, verbose=False,
)

Note that we are picking a BATCH_SIZE of 32 in this example.

### Now Test with a Real Image:

Let's try a real image batch! For this example, we will simply repeat one open-source dog image from http://www.dog.ceo:

In [ ]:
from skimage import io
from skimage.transform import resize
from matplotlib import pyplot as plt
import numpy as np

url='https://images.dog.ceo/breeds/retriever-golden/n02099601_3004.jpg'
img = resize(io.imread(url), (224, 224))
img = np.expand_dims(np.array(img, dtype=np.float32), axis=0) # Expand image to have a batch dimension
input_batch = np.array(np.repeat(img, BATCH_SIZE, axis=0), dtype=np.float32) # Repeat across the batch dimension

input_batch.shape

In [ ]:
plt.imshow(input_batch[0].astype(np.float32))

In [ ]:
resnet50_gpu = copy.deepcopy(resnet50).to("cuda").eval()

We need to move our batch onto GPU and properly format it to shape [32, 3, 224, 224]. 

In [ ]:
from torchvision.transforms import Normalize

normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
input_batch_chw = normalize(torch.from_numpy(input_batch).permute(0, 3, 1, 2))
input_batch_gpu = input_batch_chw.contiguous().to("cuda")

input_batch_gpu.shape

We can run a prediction on a batch using .forward():

In [ ]:
with torch.no_grad():
    predictions = np.array(resnet50_gpu(input_batch_gpu).cpu())

predictions.shape

### Verify Baseline Model Performance/Accuracy:

For a baseline, lets time our prediction in FP32:

In [ ]:
%%timeit

with torch.no_grad():
    preds = np.array(resnet50_gpu(input_batch_gpu).cpu())

We can also time FP16 precision performance:

In [ ]:
resnet50_gpu_half = copy.deepcopy(resnet50_gpu).half()
input_half = input_batch_gpu.half()

with torch.no_grad():
    preds = resnet50_gpu_half(input_half).cpu().numpy()

preds.shape

In [ ]:
%%timeit

with torch.no_grad():
    preds = np.array(resnet50_gpu_half(input_half).cpu())

Inspect the top five classes for this image. These values are logits, not probabilities; a single image does not establish dataset accuracy.

In [ ]:
indices = (-predictions[0]).argsort()[:5]
print("Class | Logit")
list(zip(indices, predictions[0][indices]))

We have a model exported to ONNX and a baseline to compare against! Let's now take our ONNX model and convert it to a TensorRT inference engine.

## 2. What batch size(s) am I running inference at?

The ONNX export fixes the input shape to [BATCH_SIZE, 3, 224, 224]. Use the same batch size for inference. To change it, rerun the export; a fixed shape does not require an explicit-batch command-line flag.

In [ ]:
print("Exported batch size:", BATCH_SIZE)

Runtime buffers must match the engine I/O types. Changing a NumPy array dtype does not change the precision of the operations inside an engine.

## 3. What precision am I running inference at?

USE_FP16 above selects the model and dummy input dtype before ONNX export. A strongly typed TensorRT network derives operation types from that graph. To switch to FP32, change USE_FP16 and rerun the export and engine build.

Uniform FP16 can overflow or lose accuracy. For models that need selected FP32 operations, use explicit casts or the [mixed-precision autocast sample](../../samples/python/strongly_type_autocast/) and validate representative inputs. This is not the same tactic-selection policy as the old FP16 builder flag.

In [ ]:
import numpy as np

target_dtype = np.float16 if USE_FP16 else np.float32

 To create a test batch, we will once again repeat one open-source dog image from http://www.dog.ceo:

In [ ]:
from skimage import io
from skimage.transform import resize
from matplotlib import pyplot as plt
import numpy as np

url='https://images.dog.ceo/breeds/retriever-golden/n02099601_3004.jpg'
img = resize(io.imread(url), (224, 224))
input_batch = np.array(np.repeat(np.expand_dims(np.array(img, dtype=np.float32), axis=0), BATCH_SIZE, axis=0), dtype=np.float32)

input_batch.shape

In [ ]:
plt.imshow(input_batch[0].astype(np.float32))

### Preprocess Images:

Use the same ImageNet normalization as the framework baseline, then convert the contiguous input array to the exported model dtype.

In [ ]:
def preprocess_image(img):
    result = normalize(torch.from_numpy(img).permute(2, 0, 1))
    return np.ascontiguousarray(result.numpy(), dtype=target_dtype)

preprocessed_images = np.ascontiguousarray([preprocess_image(image) for image in input_batch])

## 4. What TensorRT path am I using to convert my model?

Use the TensorRT Python builder through the helper to build a strongly typed engine from the exported ONNX model. TensorRT and CUDA Python must be installed for your CUDA environment.

In [ ]:
from onnx_helper import convert_onnx_to_engine

The model already encodes the requested precision. Pass fp16_mode=False to preserve its types; no FP16 builder flag or I/O format override is needed.

In [ ]:
serialized_engine, build_logger = convert_onnx_to_engine(
    "resnet50_pytorch.onnx", "resnet_engine_pytorch.trt", fp16_mode=False
)

The serialized engine is saved as resnet_engine_pytorch.trt.

## 5. What TensorRT runtime am I targeting?

Now, we have a converted our model to a TensorRT engine. Great! That means we are ready to load it into the native Python TensorRT runtime. This runtime strikes a balance between the ease of use of the high level Python APIs used in frameworks and the fast, low level C++ runtimes available in TensorRT.

In [ ]:
%%time

import tensorrt as trt
from cuda.bindings import runtime as cudart
import numpy as np

err, = cudart.cudaSetDevice(0)
assert err == cudart.cudaError_t.cudaSuccess

runtime_logger = build_logger
runtime = trt.Runtime(runtime_logger)
with open("resnet_engine_pytorch.trt", "rb") as f:
    engine = runtime.deserialize_cuda_engine(f.read())
assert engine is not None
context = engine.create_execution_context()
assert context is not None

Now allocate input and output memory, give TRT pointers (bindings) to it:

In [ ]:
input_name, output_name = "images", "logits"
expected_shape = tuple(engine.get_tensor_shape(input_name))
expected_dtype = np.dtype(trt.nptype(engine.get_tensor_dtype(input_name)))
assert preprocessed_images.shape == expected_shape
assert preprocessed_images.dtype == expected_dtype
output = np.empty(tuple(engine.get_tensor_shape(output_name)),
                  dtype=trt.nptype(engine.get_tensor_dtype(output_name)))

err, d_input = cudart.cudaMalloc(preprocessed_images.nbytes)
assert err == cudart.cudaError_t.cudaSuccess
err, d_output = cudart.cudaMalloc(output.nbytes)
assert err == cudart.cudaError_t.cudaSuccess
assert context.set_tensor_address(input_name, d_input)
assert context.set_tensor_address(output_name, d_output)
err, stream = cudart.cudaStreamCreate()
assert err == cudart.cudaError_t.cudaSuccess

Next, set up the prediction function.

This involves a copy from CPU RAM to GPU VRAM, executing the model, then copying the results back from GPU VRAM to CPU RAM:

In [ ]:
def predict(batch):
    if batch.shape != expected_shape or batch.dtype != expected_dtype or not batch.flags.c_contiguous:
        raise ValueError("Input must be contiguous and match the engine shape and dtype")
    err, = cudart.cudaMemcpyAsync(d_input, batch.ctypes.data, batch.nbytes,
        cudart.cudaMemcpyKind.cudaMemcpyHostToDevice, stream)
    assert err == cudart.cudaError_t.cudaSuccess
    if not context.execute_async_v3(stream):
        raise RuntimeError("TensorRT inference failed")
    err, = cudart.cudaMemcpyAsync(output.ctypes.data, d_output, output.nbytes,
        cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost, stream)
    assert err == cudart.cudaError_t.cudaSuccess
    err, = cudart.cudaStreamSynchronize(stream)
    assert err == cudart.cudaError_t.cudaSuccess
    return output

Let's time the function!

In [ ]:
print("Warming up...")

pred = predict(preprocessed_images)

print("Done warming up!")

In [ ]:
%%timeit

pred = predict(preprocessed_images)

Finally we should verify our TensorRT output is still accurate.

In [ ]:
reference_model = resnet50_gpu_half if USE_FP16 else resnet50_gpu
with torch.no_grad():
    reference = reference_model(torch.from_numpy(preprocessed_images).to("cuda")).cpu().numpy()
atol, rtol = (0.05, 0.02) if USE_FP16 else (0.001, 0.001)
np.testing.assert_allclose(pred, reference, atol=atol, rtol=rtol)
if USE_FP16:
    np.testing.assert_allclose(pred.astype(np.float32), predictions, atol=0.05, rtol=0.02)
print("Maximum absolute difference:", np.max(np.abs(pred.astype(np.float32) - reference.astype(np.float32))))
indices = (-pred[0]).argsort()[:5]
print("Class | Logit")
list(zip(indices, pred[0][indices]))

In [ ]:
err, = cudart.cudaStreamDestroy(stream)
assert err == cudart.cudaError_t.cudaSuccess
err, = cudart.cudaFree(d_input)
assert err == cudart.cudaError_t.cudaSuccess
err, = cudart.cudaFree(d_output)
assert err == cudart.cudaError_t.cudaSuccess

The assertion compares every logit against the same-precision PyTorch model on the same normalized input. Also compare against the FP32 baseline and evaluate representative data before choosing FP16 for deployment. This repeated single-image batch does not measure dataset accuracy.

## Next Steps:

<h4>  TRT Dev Docs </h4>

Main documentation page for the ONNX, layer builder, C++, and legacy APIs

You can find it here: https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html

<h4>  TRT OSS GitHub </h4>

Contains OSS TRT components, sample applications, and plugin examples

You can find it here: https://github.com/NVIDIA/TensorRT


#### TRT Supported Layers:

https://docs.nvidia.com/deeplearning/tensorrt/support-matrix/index.html#layers-precision-matrix

#### TRT ONNX Plugin Example:

https://github.com/NVIDIA/TensorRT/tree/main/samples/sampleOnnxMnistCoordConvAC